# Phase 9 — conditional Monte Carlo match simulation

This phase uses only the 76 locked test matches and the already-saved probabilities from the frozen Phase 8 global candidate. Each observed shot becomes an independent Bernoulli trial using its frozen P(goal). The simulation does not retrain a model, choose a new candidate, or predict shots that did not occur.

For every match, the notebook compares 100,000 simulated scorelines with the exact Poisson-binomial distribution for each team's observed shots. This is a conditional score simulation, not a pre-match forecast. Own goals are kept as reconciliation context because the shot model does not generate them.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / '.venv').is_dir() and (p / 'README.md').is_file()), None)
if project_root is None:
    raise RuntimeError('Could not find the project root containing .venv/ and README.md. Open this notebook from the project checkout.')
if Path(sys.prefix).resolve() != (project_root / '.venv').resolve():
    raise RuntimeError(f'Select the project .venv kernel in VS Code. Active environment: {sys.prefix}')
if not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    raise RuntimeError('The project .venv Python executable is missing.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'Project: {project_root}')
print(f'Kernel:  {sys.executable}')

## 1. Verify inputs and run the fixed simulation

The script verifies the Phase 8 protocol, saved selection, test report, and every test-output hash. It also verifies the Phase 4 hashes for `shots.csv`, `goal_reconciliation.csv`, and `own_goals.csv` before joining any simulation inputs. If an input changed, it stops instead of using an unverified probability. The example match is chosen by earliest date and match ID, before inspecting its score or predicted probabilities.

In [ ]:
import subprocess

script = project_root / 'scripts' / 'simulate_matches.py'
result = subprocess.run([sys.executable, str(script)], cwd=project_root, check=False, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Phase 9 simulation failed with exit code {result.returncode}; see the output above.')

## 2. Inspect held-out match summaries

The exact expected shot-goal total for a team is the sum of its frozen shot probabilities. The exact outcome probabilities come from the full independent-team scoreline distribution. Monte Carlo results approximate those exact values; six-MC-standard-error checks validate simulation noise, not the accuracy of the fitted football model.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

tables = project_root / 'reports' / 'tables'
summary = pd.read_csv(tables / 'phase09_match_summary.csv', float_precision='round_trip')
report = json.loads((project_root / 'docs' / 'phase_09_simulation_report.json').read_text(encoding='utf-8'))
print(f"Matches: {len(summary)} | test shots: {report['shot_count']} | trials per match: {report['trial_count_per_match']:,}")
print(f"Frozen candidate: {report['candidate']} | example match ID: {report['example_match_id']}")
columns = ['match_date', 'home_team_name', 'away_team_name', 'home_expected_shot_goals', 'away_expected_shot_goals', 'exact_home_win_probability', 'exact_draw_probability', 'exact_away_win_probability']
compact = summary.sort_values(['match_date', 'match_id'], kind='stable')[columns].rename(columns={'match_date': 'date', 'home_team_name': 'home', 'away_team_name': 'away', 'home_expected_shot_goals': 'home_xG', 'away_expected_shot_goals': 'away_xG', 'exact_home_win_probability': 'P_home', 'exact_draw_probability': 'P_draw', 'exact_away_win_probability': 'P_away'})
print('Chronological sample of 10 held-out matches; the complete 76-row table is reports/tables/phase09_match_summary.csv.')
display(compact.head(10).reset_index(drop=True))
example = summary.loc[summary.match_id == report['example_match_id']].iloc[0]
example_teams = pd.DataFrame([
    {'team': example.home_team_name, 'shots': example.home_shot_count, 'expected_goals': example.home_expected_shot_goals, 'MC_mean_goals': example.home_mc_mean_goals, 'MCSE': example.home_mean_mcse, 'expected_points': example.home_expected_points_exact},
    {'team': example.away_team_name, 'shots': example.away_shot_count, 'expected_goals': example.away_expected_shot_goals, 'MC_mean_goals': example.away_mc_mean_goals, 'MCSE': example.away_mean_mcse, 'expected_points': example.away_expected_points_exact},
])
print('Example match team summaries:')
display(example_teams)
print(f"Official score {int(example.official_home_goals)}–{int(example.official_away_goals)} | observed shot goals {int(example.observed_shot_goals_home)}–{int(example.observed_shot_goals_away)} | own goals credited home/away {int(example.own_goals_credited_home)}/{int(example.own_goals_credited_away)}")
outcomes = pd.DataFrame([
    {'outcome': 'Home win', 'exact_probability': example.exact_home_win_probability, 'MC_probability': example.mc_home_win_probability},
    {'outcome': 'Draw', 'exact_probability': example.exact_draw_probability, 'MC_probability': example.mc_draw_probability},
    {'outcome': 'Away win', 'exact_probability': example.exact_away_win_probability, 'MC_probability': example.mc_away_win_probability},
])
outcomes['absolute_difference'] = (outcomes.exact_probability - outcomes.MC_probability).abs()
display(outcomes)
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase09_example_scoreline_heatmap.png')))

The exact PMF for each team is computed by convolving `[1 - p, p]` for every observed shot. The joint scoreline grid is the outer product of the home and away PMFs. Monte Carlo samples each shot with its own independent Bernoulli draw and sums the goals by team. The exact calculation is a deterministic reference, not a substitute model.

In [ ]:
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase09_example_outcomes.png')))
display(Image(filename=str(project_root / 'reports' / 'figures' / 'phase09_example_convergence.png')))
scorelines = pd.read_csv(tables / 'phase09_scoreline_distribution.csv', float_precision='round_trip')
example_scores = scorelines.loc[scorelines.match_id == report['example_match_id']]
print(f"Top 10 example scorelines by exact probability; complete {len(example_scores)}-cell support is reports/tables/phase09_scoreline_distribution.csv.")
display(example_scores.sort_values('exact_probability', ascending=False, kind='stable').head(10)[['home_goals', 'away_goals', 'exact_probability', 'mc_count', 'mc_probability']])

## 3. Keep official results and simulated shot scorelines distinct

Official scores include own goals. The shot-only model does not sample own goals, so a simulated scoreline can differ from the official score. The match table reports official goals, observed shot goals, and own-goal credits in separate columns. Do not describe these conditional simulations as what a team 'should have scored' or as a full-season forecast.

All outputs and input hashes are listed in `docs/phase_09_simulation_report.json`; fixed seeds, trial count, and source locks are recorded in `docs/phase_09_protocol.json`.